<a href="https://colab.research.google.com/github/IsaacFigNewton/CommitteeHearingDiscourseParser/blob/main/DH2024_DigitalDemocracy_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Accessing & working with the Digital Democracy Corpus: Comprehensive Proceedings of Four State Legislatures 2015-2018


This notebook will act as a guide for a person with minimal knowledge of coding to make use of the files and retrieve useful information from the corpus.


The dataset can be downloaded directly from this website: https://huggingface.co/datasets/iatpp/digitaldemocracy-2015-2018

This corpus is distributed under the cc-by-nc-sa-4.0 license. Please review before accessing. https://creativecommons.org/licenses/by-nc-sa/4.0/

Please cite as:

Khosmood, F., Dekhtyar, A., Ellwein, S., White, B., "The Digital Democracy Corpus: Comprehensive Proceedings of Four State Legislatures 2015-2018", Digital Humanities, Washington, DC, August 2024.

# Download files

The Digital Democracy Corpus can be downloaded from here: https://huggingface.co/datasets/iatpp/digitaldemocracy-2015-2018

The following code cell will downloaded the relevant files from the data repository and store them in the Colab runtime.

 Files in Colab runtime storage can be accessed by clicking on the folder button on the left sidebar. Double click on a file to view a preview of the file. Anything stored Colab runtime storage will disappear once the session is over. To save these files to your physical computer, right click on the file you'd like to save and click download.

In [ ]:
from pathlib import Path
import subprocess
import zipfile
import os

#This is where the unzipped corpus file is stored
CORPUS_FILE_PATH = 'DH2024_Corpus_Release/'
corpus_dir = Path(CORPUS_FILE_PATH)
zip_path = Path("digitaldemocracy-2015-2018/DH2024_Corpus_Release.zip")
repo_dir = Path("digitaldemocracy-2015-2018")

# Clone repository if not present
if not repo_dir.is_dir():
    print("Corpus directory not found. Cloning repository...")
    subprocess.run(
        ["git", "clone", "https://huggingface.co/datasets/iatpp/digitaldemocracy-2015-2018"],
        check=True,
    )
    print("Repository cloned successfully.")

# Extract zip file if corpus directory doesn't exist
if not corpus_dir.is_dir():
    if zip_path.exists():
        print(f"Extracting {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('.')
        print("Extraction complete.")
    else:
        print(f"Error: {zip_path} not found!")
else:
    print(f"Corpus already extracted at {corpus_dir}")

# Import and config

In [ ]:
from src import HearingLoader, HearingTagger, HearingParser

text_features = [HearingParser.TEXT_COL]
categorical_features = HearingParser.CAT_COLS
numeric_features = HearingParser.NUM_COLS

# Initialize the HearingLoader with the corpus path
loader = HearingLoader(corpus_path='DH2024_Corpus_Release/')
tagger = HearingTagger()
parser = HearingParser()

# Discussion Transcripts

In [ ]:
# discussion will contain all the information from bill CA_201720180AB10 in hearing id 51835
# 'CA_201720180AB10': ['51835',
#   '52298',
#   '52708',
#   '52856',
#   '53960',
#   '54363',
#   '54505']
hearing = loader.bill_discussion_info(51835, "CA_201720180AB10")
# vars(hearing)

In [ ]:
# Using pprint_hearing we layout the transcript in a human readable format
HearingLoader.pprint_hearing(hearing)

# Try loading all the hearings

In [ ]:
hearings = loader.load_all_committee_hearings()
relevant_hearings = [h for h in hearings if h.bid != 'CA_NO BILL DISCUSSED']

In [ ]:
print(len(hearings))
print(len(relevant_hearings))

In [ ]:
relevant_hearings = [tagger(h) for h in relevant_hearings]

In [ ]:
utterances_df = parser._build_utterance_rows(relevant_hearings)

In [ ]:
utterances_df.head(500).to_csv('utterances_sample.csv')

In [ ]:
import pandas as pd

labeled_utterances_df = pd.read_csv('./utterances_sample_labeled.csv')

In [ ]:
labeled_utterances_df.head()

In [ ]:
labeled_utterances_df['stage_label'].value_counts()

In [ ]:
for col in text_features:
  labeled_utterances_df[col] = labeled_utterances_df[col].fillna('')

labeled_utterances_df['stage_label'] = labeled_utterances_df['stage_label'].fillna('unknown')
labeled_utterances_df["speaker_position"] = labeled_utterances_df["speaker_position"]/labeled_utterances_df["speaker_position"].max()

# Evaluation

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

groups = labeled_utterances_df['bid']

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(labeled_utterances_df, labeled_utterances_df['stage_label'], groups)
)

train_df = labeled_utterances_df.iloc[train_idx].copy()
test_df = labeled_utterances_df.iloc[test_idx].copy()

print('Train rows:', len(train_df))
print('Test rows:', len(test_df))

In [ ]:
for col in numeric_features:
  labeled_utterances_df[col] = labeled_utterances_df[col].fillna(0)
  train_df[col] = train_df[col].fillna(0)
  test_df[col] = test_df[col].fillna(0)

for col in categorical_features:
  labeled_utterances_df[col] = labeled_utterances_df[col].fillna('unknown')
  train_df[col] = train_df[col].fillna('unknown')
  test_df[col] = test_df[col].fillna('unknown')

In [ ]:
# Prepare train and test data
feature_cols = text_features + categorical_features + numeric_features

X_train = train_df[feature_cols]
y_train = train_df['stage_label']

X_test = test_df[feature_cols]
y_test = test_df['stage_label']

# Train the model using HearingParser's train_model method
parser.train_model(train_df, label_col='stage_label')

# The model is now accessible via parser.model

In [ ]:
from sklearn.metrics import classification_report

# Use the parser's model for prediction
pred = parser.model.predict(X_test)

print(classification_report(
    y_test,
    pred,
    digits=3,
))

In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import ConfusionMatrixDisplay


labels = sorted(y_test.unique())

fig, ax = plt.subplots(figsize=(10, 8))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred,
    labels=labels,
    xticks_rotation=45,
    ax=ax,
)

ax.set_title('Stage Classifier Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Removed standalone functions - now using HearingParser methods:
# - parser.smooth_label_list() for label smoothing
# - parser.predict_hearing_sections() for single hearing prediction
# - parser.predict_hearings_batch() for batch prediction

In [ ]:
# Use the parser's predict_hearing_sections method
labels = parser.predict_hearing_sections(
    hearing=relevant_hearings[0],
    utterances_df=utterances_df,
    smooth=True,
)

print(labels)

In [ ]:
special_hearing = None
for h in relevant_hearings:
  if h.bid == 'CA_201720180AB2721':
    special_hearing = h
    break

In [ ]:
standard_example = relevant_hearings[0]

Note: special case of Chairman presenting to someone else: Bill ID CA_201720180SB1293

Special kind of bill discussion: CA_201720180AB1708
Case where you have to use handoff cue: CA_201720180AB2721
CA_201720180SCR159

In [ ]:
def pprint_hearing_labels(hearing, parser, utterances_df):
  """Print formatted transcript with predicted section labels.
  
  Args:
      hearing: The hearing to print
      parser: HearingParser instance with trained model
      utterances_df: DataFrame with utterance features
  """
  labels = parser.predict_hearing_sections(
    hearing=hearing,
    utterances_df=utterances_df,
    smooth=True,
  )

  print()
  print(f"State:\t\t{hearing.state}")
  print(f"Committee:\t{hearing.cname}")
  print(f"Bill:\t\t{hearing.bid}")
  print(f"Date:\t\t{hearing.hearing_date.strftime('%Y-%m-%d')}")
  print()
  print("Transcript:")
  for i, contribution in enumerate(hearing.utterances):
    speaker = hearing.speakers[contribution.pid]
    print(f'[{labels[i] if i < len(labels) else "UNKNOWN"}]')
    first_name = speaker.first_name or "UNKNOWN"
    last_name = speaker.last_name or "UNKNOWN"
    name = f"{first_name} {last_name}:"
    print(f"{name:<20} {contribution.text}")
  print()

# Sanity Checks

In [ ]:
standard_example_2 = relevant_hearings[17]

In [ ]:
HearingLoader.pprint_hearing(standard_example)

In [ ]:
pprint_hearing_labels(standard_example, parser, utterances_df)

In [ ]:
len(relevant_hearings[1].utterances)

In [ ]:
tagger._detect_first_presentation_utterance(relevant_hearings[9])

In [ ]:
relevant_hearings[4].bid